# Master evaluation — every canonical run, scored the identical way

**The rule that makes this reviewable: no metric math lives in this notebook.** Every number
is a one-line call into `pim.*` — probes from `pim.probes`, editors from `pim.editors`,
scores from `pim.metrics`, benches from `pim.environments.*` — so reviewing those packages
*is* reviewing these numbers. This notebook only: scans runs, calls in, writes `scores.json`.

**Flow.** [1] scan `runs/**/config.json` (excluding `runs/archive/`, hardcoded) → run table ·
[2] the canonical evaluation settings, in one visible place · [3]/[4] the per-environment
scorers (thin wiring) · [5] score every run whose `scores.json` is missing or stamped with a
stale `EVAL_VERSION`, and write it into the run dir · [6] per-run summary tables.

**What a `scores.json` holds.** Decodability (Probe Skill — the cross-environment axis — plus
native R² / error-rate, per residual point, with the MLP ≥ linear tripwire report), held-out
gates (Othello: legal mass, top-1, CE excess over the exact Bayes floor), and editability
(the three workhorse editors PI / ND / GS, full sweep arrays plus each editor's best arm with
its guards: fidelity, collateral, li-error-vs-pre). Oracle editors and the nullspace
probe/editor are a dedicated opt-in analysis, not part of this default loop.

Companion: `build_full_table.ipynb` reads every `scores.json` this writes and renders the
single master table.

In [ ]:
# [1] Scan runs/ (recursively) -> the run table. Two exclusions, both by convention:
#     runs/archive/ (hardcoded, per the housecleaning rules) and any topic dir starting
#     with "_" (private/scratch, e.g. runs/_smoke — score those by renaming the topic).
import json, os, sys, time
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)                       # every pim loader resolves datasets/ against the repo root
sys.path.insert(0, str(REPO))

import torch
from pim.models import load_checkpoint, n_points

DEV = "cuda" if torch.cuda.is_available() else "cpu"

def scan_runs(root: Path = REPO / "runs") -> list[dict]:
    rows = []
    for cfg_path in sorted(root.rglob("config.json")):
        rel = cfg_path.relative_to(root)
        if rel.parts[0] == "archive" or rel.parts[0].startswith("_"):
            continue
        cfg = json.loads(cfg_path.read_text())
        run_dir = cfg_path.parent
        if not (run_dir / "best_model.pt").exists():
            continue                                     # not a run dir (stray config)
        rows.append({
            "topic": str(rel.parts[0]), "run": run_dir.name, "dir": run_dir,
            "arch": cfg.get("arch", "?"),
            "env": cfg.get("data", {}).get("env", "?"),
            "instance": cfg.get("data", {}).get("instance", "?"),
            "n_params": cfg.get("n_params"),
            "scored": (run_dir / "scores.json").exists(),
        })
    return rows

RUNS = scan_runs()
print(f"{'topic':<28} {'run':<16} {'arch':<22} {'env':<10} {'instance':<12} scored")
for r in RUNS:
    print(f"{r['topic']:<28} {r['run']:<16} {r['arch']:<22} {r['env']:<10} "
          f"{r['instance']:<12} {r['scored']}")

In [ ]:
# [2] The canonical evaluation settings — every knob, in one visible place.
# Bump EVAL_VERSION to force a rescore of every run (a changed setting or a fixed bug);
# a run is skipped iff its scores.json carries this exact version.
EVAL_VERSION = "2026-09-01.4"   # discworld now fits ONE probe set per basis (the FULL
                                # state) and sweeps editability over BOTH dim sets,
                                # reporting the better. (.3 fitted pos and full probes
                                # separately and scored them as two blocks.)

SETTINGS = {
    # -- probes ----------------------------------------------------------------
    "dw_probe_seqs": 30_000,      # from the instance's probe split (120k available);
                                  # keeps MLP-128 at ~14 rows/param, out of memorisation
    "oth_probe_games": 20_000,    # the probe index range [91M, 91.02M)
    # -- discworld editability (the 2026-08-22 spec: both axes swept) ----------
    "dw_bench_n": 192,
    # ONE probe target. The retired pos-only probe is not lost: for the LINEAR probe the
    # position rows of a full-state lstsq fit are BIT-IDENTICAL to a position-only fit
    # (multi-output least squares decomposes per output dim — verified on cached probes,
    # max|W_full[:4] − W_pos| = 0.0 in both bases), so dims="pos" reproduces it exactly.
    # The MLP does not decompose, so for GS the two dim sets are genuinely different
    # edits — which is why both are swept rather than one assumed. 2026-09-01.
    "dw_target": "full",
    "dw_edit_dims": ("pos", "all"),   # pim.environments.discworld.bench.DIM_SETS
    # A basis is a different PROBE TARGET, hence its own scored block and table row.
    # 'frustum' = u = x/(scale*y) laterally, 1/y in depth (frustum.CANONICAL_DEPTH),
    # settled by the 2026-09-01 depth pilot: the inverse-depth family beats cartesian
    # on BOTH position and velocity, while y/rho beat it on position but LOSE on
    # velocity — coordinates the model cannot observe do not help.
    "dw_bases": ("cartesian", "frustum"),
    "dw_alpha_nd": (0.05, 0.1, 0.2, 0.35, 0.5, 0.75, 1.0, 1.5, 2.0),
    "dw_alpha_pi": (0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 5.0, 8.0,
                    12.0, 20.0, 35.0, 60.0, 100.0, 175.0),
    "dw_alpha_gs": (0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.35, 0.5),
    "dw_gs_steps": 100,
    "dw_gs_beta": 0.2,
    # -- othello editability (step-0 over the 1001-case bench) -----------------
    "oth_alpha_nd": (0.05, 0.1, 0.2, 0.35, 0.5, 0.75, 1.0, 1.5, 2.0),
    "oth_alpha_pi": (0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0),
    "oth_gs_layers": (0, 2, 4, 6, 8),        # GS is the expensive editor (sequential
    "oth_alpha_gs": (0.05, 0.2, 0.35, 0.7),  # descent, ~12-15 min per 20-arm sweep)
    "oth_gs_steps": 100,
    "oth_gs_beta": 0.2,               # their reg_strg
    "oth_gates_games": 10_000,        # the whole test split
}
print(f"EVAL_VERSION {EVAL_VERSION}")

In [ ]:
# [3] The discworld scorer — thin wiring over pim.environments.discworld.bench.
#     Fitted probes are STORED IN THE RUN DIR (runs/<topic>/<run>/probes/).
#     ⛔ ND is computed but NOT REPORTED for discworld (2026-09-01): one fixed direction
#     with a swept scalar is coherent only for a CATEGORICAL target (Othello: flip a
#     tile, same change every case), never for 192 teleports of differing distance and
#     direction. Arms stay in scores.json as a record; the table omits them.
#     ⛔ The bench AND probe corpus come from the RUN'S OWN instance — a model is always
#     scored on the world it was trained in.
#     A BASIS is a different probe target, so it gets its own block and its own table
#     ROW — never a column that exists for only some runs.
#     ONE probe set per basis (the FULL state); each editor is swept over BOTH dim sets
#     ("pos" = position read-outs only, "all" = the whole state) and the BETTER arm is
#     the reported one, with the winning dim set recorded on it. See SETTINGS for why
#     this loses nothing relative to the retired pos-only probes.
import numpy as np
from pim.environments.discworld import bench as dwb
from pim.probes.mlp import check_probe_sanity

def best_arm(recs):
    b = max(recs, key=lambda r: r["edit_index"])
    return {k: v for k, v in b.items() if np.isscalar(v)}

def score_discworld(model, run_dir, s=SETTINGS) -> dict:
    probe_dir = run_dir / "probes"
    inst = (json.loads((run_dir / "config.json").read_text())
            .get("data", {}).get("instance", "dw-pn04"))
    inst_root = REPO / "datasets" / "discworld" / inst
    eval_dir, probe_corpus = inst_root / "eval", inst_root / "probe"
    target, dimsets = s["dw_target"], s["dw_edit_dims"]
    out = {"probe_dir": str(probe_dir), "instance": inst, "target": target,
           "edit_dims": list(dimsets), "bases": {}}
    for basis in s["dw_bases"]:
        b = dwb.load_bench(model, n=s["dw_bench_n"], target=target,
                           basis_name=basis, data_dir=eval_dir)
        lin = dwb.fit_probes(model, target=target, n_seq=s["dw_probe_seqs"],
                             family="linear", basis_name=basis,
                             data_dir=probe_corpus, cache_dir=probe_dir, log=None)
        mlp = dwb.fit_probes(model, target=target, n_seq=s["dw_probe_seqs"],
                             family="mlp", basis_name=basis,
                             data_dir=probe_corpus, cache_dir=probe_dir, log=None)
        sanity = check_probe_sanity(lin, mlp, strict=False, log=print, label=basis)
        u = dwb.unsteered(model, b)
        arms = []
        for dims in dimsets:
            for ell in range(n_points(model)):
                arms += dwb.nanda_arm(model, b, lin[ell][0], ell, s["dw_alpha_nd"],
                                      dims=dims)
            arms += dwb.pinv_arm(model, b, lin, s["dw_alpha_pi"], space="zspace",
                                 dims=dims)
            arms += dwb.grad_steer_arm(model, b, mlp, range(n_points(model)),
                                       s["dw_alpha_gs"], n_steps=s["dw_gs_steps"],
                                       beta=s["dw_gs_beta"], dims=dims)
        for r in arms:
            r["fidelity_ratio"] = dwb.fidelity_ratio(r, u)
        def pick(ed, dims=None):
            sub = [r for r in arms if r["editor"].startswith(ed)
                   and (dims is None or r["dims"] == dims)]
            return best_arm(sub) if sub else None
        block = {
            "unedited": {k: v for k, v in u.items() if np.isscalar(v)},
            "probe_skill_linear": [v[1]["r2"] for v in lin.values()],
            "probe_skill_mlp": [v[1]["r2"] for v in mlp.values()],
            "probe_perdim_linear": [v[1]["per_dim_r2"] for v in lin.values()],
            "probe_perdim_mlp": [v[1]["per_dim_r2"] for v in mlp.values()],
            "probe_sanity": sanity,
            # 'best' = the better of the two dim sets; the winner carries its own
            # "dims" field, so the reported number always says which edit produced it.
            "best": {ed: pick(ed) for ed in ("PI", "ND", "GS")},
            "best_by_dims": {d: {ed: pick(ed, d) for ed in ("PI", "ND", "GS")}
                             for d in dimsets},
            "arms": [{k: v for k, v in r.items() if np.isscalar(v)} for r in arms],
        }
        out["bases"][basis] = block
        pi = block["best"]["PI"]
        print(f"    {basis}: skill lin {max(block['probe_skill_linear']):+.4f}"
              f"  PI {pi['edit_index']:+.4f}/{pi['fidelity_ratio']:.2f}"
              f" (dims={pi['dims']})", flush=True)
    return out

print("discworld scorer ready (one full-state probe set per basis; dims swept)")

In [ ]:
# [4] The Othello scorer — thin wiring over pim.environments.othello.{arms,bench,corpus}.
#     As on the discworld side, fitted probes live in the run's own probes/ dir.
#     EVERYTHING is mine/theirs, sequence-split (settled 2026-09-01): once the GS
#     target-frame bug was fixed, every editor's best arm read mine/theirs probes, so
#     absolute-colour ("state") probes have no consumer; and the frame split measured
#     0.976 vs sequence's 0.975, so the honest split costs nothing. The grid is 18 fits,
#     down from 72. PROBE_SOURCES still travels into scores.json — one frame is not a
#     reason to stop recording which probes an editor read.
from pim.environments.othello import arms as oa
from pim.environments.othello import corpus as oc
from pim.environments.othello import case_targets, load_benchmark
from pim.environments.othello.data import tokens_and_labels, canonical_vocab
from pim.metrics.othello_moves import move_fidelity_ratio

PROBE_SOURCES = {
    "PI": "mine|linear|sequence",
    "ND": "mine|linear|sequence (target−current contrast; see note)",
    "GS": "mine|mlp|sequence (targets: mine/theirs via case_targets)",
}
# ND is the target−current CONTRAST direction (was "ND-sub"), canonical 2026-09-01:
# it beat the plain target-row form on every arm (+0.622 vs +0.447, fid 0.23 vs 0.34)
# and is the more principled direction — it raises the target class while lowering the
# current one, rather than raising the target alone.

def _probe_games(n):
    tok, ln = oc.load(oc.build(oc.LADDER["D"], log=lambda s: None, only=("probe",))["probe"])
    itos = {v: k for k, v in canonical_vocab().items()}
    return tokens_and_labels([[itos[int(t)] for t in row[:L]]
                              for row, L in zip(tok[:n], ln[:n])])

def score_othello(model, run_dir, s=SETTINGS) -> dict:
    probe_dir = run_dir / "probes"
    # held-out gates on the full test split (Bayes floors are exact for this generator)
    tok, ln = oc.load(oc.build(oc.LADDER["D"], log=lambda s_: None, only=("test",))["test"])
    g = oa.gates(model, tok[: s["oth_gates_games"]], ln[: s["oth_gates_games"]], log=None)

    data = _probe_games(s["oth_probe_games"])
    grid = oa.fit_probe_grid(model, data, cache_dir=probe_dir, log=None)
    # decodability: skill = 1 − err/majority_err, both from the SAME fit's train split
    skill = {}
    for st in grid.stats:
        key = (st["target"], st["family"], st["split"])
        skill.setdefault(key, []).append(
            1.0 - st["error_rate"] / st["majority_class_error_rate"])

    bench = load_benchmark()
    cur, tgt = case_targets(bench)
    u = oa.unsteered(model, bench)
    uns_probs = oa.unsteered_probs(model, bench)   # the guard's denominator
    npnt = n_points(model)
    lin_mine = {p: grid.probes[("mine", "linear", "sequence", p)] for p in range(npnt)}
    mlp_mine = {p: grid.probes[("mine", "mlp", "sequence", p)] for p in range(npnt)}
    arms_out = []
    for mode, label, alphas in (("add_sub", "ND", s["oth_alpha_nd"]),
                                ("pinv", "PI", s["oth_alpha_pi"])):
        for ell in range(npnt):
            for a in alphas:
                pr, card = oa.linear_arm(model, bench, lin_mine, tgt, cur,
                                         mode=mode, alpha=a, points={ell})
                arms_out.append({"editor": label, "point": ell, "alpha": a,
                                 "fidelity_ratio": move_fidelity_ratio(
                                     pr, uns_probs, bench.legal_post),
                                 **{k: v for k, v in card.items()
                                    if isinstance(v, (int, float))}})
    # GS steers the mine/theirs probes toward mine-coordinate targets. The frames MUST
    # match: feeding absolute-colour labels to these probes is the 2026-08-31 bug, worth
    # 0.70 Edit Index.
    for ls in s["oth_gs_layers"]:
        for a in s["oth_alpha_gs"]:
            pr, card = oa.grad_steer_arm(model, bench, mlp_mine, ls, alpha=a,
                                         n_steps=s["oth_gs_steps"],
                                         beta=s["oth_gs_beta"], target_labels=tgt)
            arms_out.append({"editor": "GS", "point": ls, "alpha": a,
                             "fidelity_ratio": move_fidelity_ratio(
                                 pr, uns_probs, bench.legal_post),
                             **{k: v for k, v in card.items()
                                if isinstance(v, (int, float))}})
    def best(ed):
        sub = [r for r in arms_out if r["editor"] == ed]
        return max(sub, key=lambda r: r["edit_index_union"]) if sub else None
    return {
        "gates": g,
        "probe_dir": str(probe_dir),
        "probe_sources": PROBE_SOURCES,
        "probe_skill": {"|".join(k): v for k, v in skill.items()},
        "probe_stats": [{k: v for k, v in st.items() if not isinstance(v, list)}
                        for st in grid.stats],
        "unedited": {**{k: v for k, v in u.items() if isinstance(v, (int, float))},
                     "fidelity_ratio": 1.0},
        "best": {ed: best(ed) for ed in ("PI", "ND", "GS")},
        "arms": arms_out,
    }

print("othello scorer ready (mine/theirs + sequence split throughout)")

In [ ]:
# [5] BASELINES — the two decodability floors, per ENVIRONMENT INSTANCE x ARCHITECTURE.
#     They live in runs/_baselines/<instance>/ — the "_" prefix is ALREADY the marker
#     scan_runs and build_full_table use to skip a directory, so baselines sit in runs/
#     without ever being mistaken for a trained run.
#
#     observation  = the SAME probes fitted to the causal INPUT HISTORY instead of the
#                    residual stream. "How much does a shallow read of the input give?"
#     random-init  = the SAME architecture, seeded, never trained, probed identically.
#                    "How much comes from training rather than from random features?"
#     Matched to the model probes in every other respect — corpus, n_seq, the identical
#     seeded 80/20 split BY SEQUENCE, targets, bases, families — so a baseline row and a
#     model row are the same measurement on different features.
#
#     observation_large (b3, 2026-09-02) = the observation floor again on the instance's
#     `probe_large` split (discworld 250k sequences, Othello 170k games — ~5x the rows),
#     50 epochs (>= 2x the canonical step count). The wide-input observation probe
#     MEMORISES the canonical 30k corpus (in-sample gap +0.25 to +0.46 on discworld), so
#     its matched-size floor is an under-estimate; the probe-capacity sweep showed the 5x
#     corpus fixes that (gap ~0.02) and lifts discworld's MLP-128 floor from 0.70 to 0.88.
#     Table 3 reports observation_large where it exists. Model probes are unaffected: at
#     ~14 rows/param they reproduce their 30k values on 250k to within 0.005.
#
#     ⛔ BOTH floors are per (instance, ARCH), not per instance (fixed 2026-09-01).
#     Random-init obviously so. Observation too, less obviously: the probe is given the
#     history the model actually consumes, and `state_span` is architecture-dependent
#     (transformer_l = block_size 39; transformer_s = n_layers*(window-1)+1). Keying
#     baselines on the instance alone would have silently compared a Transformer-S run
#     against a Transformer-L floor the moment a second architecture was trained on an
#     existing instance — which is exactly what is planned next.
#     Both kinds share ONE probes/ dir per instance: every cache key already carries the
#     model fingerprint (and, for the model-free observation probes, the span), so two
#     architectures cannot collide, and a fit shared between them is computed once.
import subprocess

from pim.probes.baselines import random_init_model

BASELINES_DIR = REPO / "runs" / "_baselines"
BASELINE_SEED = 0
BASELINE_VERSION = "2026-09-02.b3"   # b3 = + observation_large (5x corpus, 50 epochs);
                                     # b2 = per-(instance, arch); b1 = per-instance.
                                     # Deliberately SEPARATE from EVAL_VERSION: a change
                                     # to the editor sweep must not invalidate a floor,
                                     # and vice versa.
LARGE = {"dw_n_seq": 250_000, "oth_split": "probe_large", "epochs": 50}

def _sha():
    try:
        return subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True,
                              text=True, timeout=10).stdout.strip() or "unknown"
    except Exception:
        return "unknown"

def _skill(st):
    """Probe Skill from a stats dict — SELECTS the right one, computes nothing new:
    regression skill IS R² against the train mean; classification is 1 − err/majority."""
    if "r2" in st:
        return st["r2"]
    return 1.0 - st["error_rate"] / st["majority_class_error_rate"]

def _pack(st):
    """skill + the OVERFIT CHECK. Every fit reports its in-sample counterpart, and
    `insample_gap` is train minus held-out on the skill scale: it is how Table 3 says
    whether a wide-input probe memorised its train split instead of learning the map.
    Model probes sit at +0.000 to +0.005."""
    gap = (st["r2_insample"] - st["r2"] if "r2" in st else
           (st["error_rate"] - st["error_rate_insample"]) / st["majority_class_error_rate"])
    return {"skill": _skill(st), "insample_gap": gap, "d_in": st.get("d_in"),
            "n_train_rows": st.get("n_train_rows")}

def _agg(per_pt):
    """A random-init model has residual points like any other; report its best, the way
    every model row reports its best point."""
    best = max(range(len(per_pt)), key=lambda i: per_pt[i]["skill"])
    return {**per_pt[best], "point": best, "per_point": per_pt}

def score_baselines_arch(inst, env, arch, model_config, s=SETTINGS) -> dict:
    """Both floors for ONE (instance, architecture) pair."""
    pdir = BASELINES_DIR / inst / "probes"
    rand = random_init_model(arch, model_config, seed=BASELINE_SEED, device=DEV)
    span = int(getattr(rand, "state_span", 39))
    block = {"span": span, "bases": {}}
    if env == "discworld":
        inst_root = REPO / "datasets" / "discworld" / inst
        probe_corpus, large = inst_root / "probe", inst_root / "probe_250k"
        for basis in s["dw_bases"]:
            blk = {"observation": {}, "random_init": {}, "observation_large": {}}
            for fam in ("linear", "mlp"):
                # observation probes carry NO model; the span is in their cache key, so
                # architectures that share a span share the fit
                _, st = dwb.observation_probes(
                    target=s["dw_target"], n_seq=s["dw_probe_seqs"], family=fam,
                    basis_name=basis, span=span, data_dir=probe_corpus,
                    cache_dir=pdir, log=print)
                blk["observation"][fam] = _pack(st)
                if (large / "test.h5").exists():
                    _, st = dwb.observation_probes(
                        target=s["dw_target"], n_seq=LARGE["dw_n_seq"], family=fam,
                        basis_name=basis, span=span, data_dir=large, cache_dir=pdir,
                        log=None, epochs=LARGE["epochs"])
                    blk["observation_large"][fam] = {**_pack(st), "n_seq": LARGE["dw_n_seq"],
                                                     "epochs": LARGE["epochs"]}
                # The random-init floor needs a different MODEL, never a different
                # measurement — this is the ordinary probe path, entirely unchanged.
                fits = dwb.fit_probes(rand, target=s["dw_target"],
                                      n_seq=s["dw_probe_seqs"], family=fam,
                                      basis_name=basis, data_dir=probe_corpus,
                                      cache_dir=pdir, log=None)
                blk["random_init"][fam] = _agg([_pack(v[1]) for v in fits.values()])
                ol = blk["observation_large"].get(fam, {}).get("skill", float("nan"))
                print(f"    {arch}/{basis}/{fam}: obs {blk['observation'][fam]['skill']:+.4f}"
                      f"  obs_large {ol:+.4f}  random-init "
                      f"{blk['random_init'][fam]['skill']:+.4f}", flush=True)
            block["bases"][basis] = blk
    else:
        data = _probe_games(s["oth_probe_games"])
        grid = oa.fit_probe_grid(rand, data, cache_dir=pdir, log=None)
        paths = oc.build(only=(LARGE["oth_split"],), log=lambda s_: None)
        data_large = oc.probe_data(paths[LARGE["oth_split"]])      # labels cached beside the corpus
        blk = {"observation": {}, "random_init": {}, "observation_large": {}}
        for fam in ("linear", "mlp"):
            _, st = oa.observation_probes(data, family=fam, seed=BASELINE_SEED,
                                          cache_dir=pdir, log=print)
            blk["observation"][fam] = _pack(st)
            _, st = oa.observation_probes(data_large, family=fam, seed=BASELINE_SEED,
                                          cache_dir=pdir, log=None, epochs=LARGE["epochs"])
            blk["observation_large"][fam] = {**_pack(st), "n_seq": int(len(data_large.tokens)),
                                             "epochs": LARGE["epochs"]}
            pts = [x for x in grid.stats if x["target"] == "mine"
                   and x["family"] == fam and x["split"] == "sequence"]
            blk["random_init"][fam] = _agg([_pack(x) for x in pts])
            print(f"    {arch}/mine/{fam}: obs {blk['observation'][fam]['skill']:+.4f}"
                  f"  obs_large {blk['observation_large'][fam]['skill']:+.4f}"
                  f"  random-init {blk['random_init'][fam]['skill']:+.4f}", flush=True)
        block["bases"]["mine/theirs"] = blk
    del rand
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return block

# every (instance, arch) pair that has a trained run — an instance may carry several
INSTANCES = {}
for r in RUNS:
    key = r["instance"]
    if key not in INSTANCES:
        INSTANCES[key] = {"env": r["env"], "archs": {}}
    if r["arch"] not in INSTANCES[key]["archs"]:
        _m, _i = load_checkpoint(r["dir"] / "best_model.pt", device="cpu")
        INSTANCES[key]["archs"][_i.arch] = _i.model_config
        del _m

for inst, spec in INSTANCES.items():
    bp = BASELINES_DIR / inst / "baselines.json"
    prev = json.loads(bp.read_text()) if bp.exists() else {}
    fresh = prev.get("baseline_version") == BASELINE_VERSION
    have = set(prev.get("archs", {})) if fresh else set()
    todo = [a for a in spec["archs"] if a not in have]
    if not todo:
        print(f"skip  baselines/{inst}  ({BASELINE_VERSION}, archs {sorted(have)})")
        continue
    t0 = time.time()
    print(f"\n=== baselines for {inst} ({spec['env']}) archs {todo} ===", flush=True)
    out = prev if fresh else {"instance": inst, "env": spec["env"],
                              "seed": BASELINE_SEED, "archs": {}}
    out.setdefault("archs", {})
    for arch in todo:
        out["archs"][arch] = score_baselines_arch(inst, spec["env"], arch,
                                                  spec["archs"][arch])
    out |= {"baseline_version": BASELINE_VERSION, "commit_sha": _sha(),
            "minutes": round((time.time() - t0) / 60, 1)}
    bp.parent.mkdir(parents=True, exist_ok=True)
    bp.write_text(json.dumps(out, indent=1, default=float))
    print(f"    wrote {bp.relative_to(REPO)}  [{out['minutes']} min]", flush=True)

print("\nall baselines present")

In [ ]:
# [6] Score every run whose scores.json is missing or stamped with a stale EVAL_VERSION.
#     (`_sha` comes from [5]; the baselines it wrote are per-instance and already cached.)
SCORERS = {"discworld": score_discworld, "othello": score_othello}

for r in RUNS:
    sp = r["dir"] / "scores.json"
    if sp.exists():
        prev = json.loads(sp.read_text())
        if prev.get("eval_version") == EVAL_VERSION:
            print(f"skip  {r['topic']}/{r['run']}  (scored at {EVAL_VERSION})")
            continue
        print(f"stale {r['topic']}/{r['run']}  ({prev.get('eval_version')} -> {EVAL_VERSION})")
    if r["env"] not in SCORERS:
        print(f"skip  {r['topic']}/{r['run']}  (env {r['env']!r} has no scorer)")
        continue
    t0 = time.time()
    print(f"\n=== scoring {r['topic']}/{r['run']}  ({r['arch']} on {r['env']}) ===", flush=True)
    model, info = load_checkpoint(r["dir"] / "best_model.pt", device=DEV)
    scores = SCORERS[r["env"]](model, r["dir"])
    scores = {"run": f"{r['topic']}/{r['run']}", "arch": info.arch,
              "env": r["env"], "instance": r["instance"],
              "val_loss": info.val_loss, "n_points": n_points(model),
              "eval_version": EVAL_VERSION, "commit_sha": _sha(),
              "settings": {k: (list(v) if isinstance(v, tuple) else v)
                           for k, v in SETTINGS.items()},
              "minutes": round((time.time() - t0) / 60, 1), **scores}
    sp.write_text(json.dumps(scores, indent=1, default=float))
    print(f"    wrote {sp.relative_to(REPO)}  [{scores['minutes']} min]", flush=True)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nall runs scored")

In [ ]:
# [6] Per-run summaries: the headline block of each scores.json, human-readable.
#     Discworld prints one block per BASIS; each editor's reported arm names the dim set
#     that won it, and the losing dim set is shown beside it so the choice is visible
#     rather than buried in scores.json.
for r in RUNS:
    sp = r["dir"] / "scores.json"
    if not sp.exists():
        continue
    s = json.loads(sp.read_text())
    print(f"\n{'=' * 86}\n{s['run']}   ({s['arch']} on {s['env']}/{s['instance']})   "
          f"val {s['val_loss']:.5f}\n{'=' * 86}")
    if s["env"] == "discworld":
        for basis, T in s["bases"].items():
            u = T["unedited"]
            print(f"  basis={basis}  target={s.get('target', 'full')}  "
                  f"UNEDITED EI {u['edit_index']:+.4f}   "
                  f"probe skill (linear, best point) {max(T['probe_skill_linear']):+.4f}  "
                  f"(mlp) {max(T['probe_skill_mlp']):+.4f}  "
                  f"tripwire violations {T['probe_sanity']['n_violations']}")
            print(f"  {'editor':<8}{'dims':>6}{'pt':>4}{'alpha':>8}{'EI':>9}{'fid':>7}"
                  f"{'target':>9}{'collat':>9}   | EI by dim set")
            for ed, bst in T["best"].items():
                if not bst:
                    continue
                byd = "  ".join(
                    f"{d}={T['best_by_dims'][d][ed]['edit_index']:+.4f}"
                    for d in s.get("edit_dims", []) if T["best_by_dims"][d].get(ed))
                print(f"  {ed:<8}{bst.get('dims', '—'):>6}{bst['point']:>4}"
                      f"{bst['alpha']:>8}{bst['edit_index']:>+9.4f}"
                      f"{bst['fidelity_ratio']:>7.3f}{bst['target_rmse']:>9.4f}"
                      f"{bst['collateral_rmse']:>9.4f}   | {byd}")
    else:
        g = s["gates"]
        print(f"  gates: legal mass {g['legal_mass']:.4f}  top-1 legal {g['top1_legal']:.4f}  "
              f"CE {g['ce']:.4f} (Bayes {g['bayes_ce']:.4f}, excess {g['ce'] - g['bayes_ce']:+.4f})")
        sk = s["probe_skill"]
        for key in ("mine|linear|sequence", "mine|mlp|sequence"):
            if key in sk:
                print(f"  probe skill [{key}]: best point {max(sk[key]):+.4f}")
        u = s["unedited"]
        print(f"  UNEDITED EI(union) {u['edit_index_union']:+.4f}   li vs post {u['li_error_vs_post']:.3f}")
        print(f"  {'editor':<12}{'pt':>4}{'alpha':>8}{'EI(un)':>9}{'li post':>9}{'li pre':>9}{'legal':>8}")
        for ed, bst in s["best"].items():
            if bst:
                print(f"  {ed:<12}{bst['point']:>4}{bst['alpha']:>8}"
                      f"{bst['edit_index_union']:>+9.4f}{bst['li_error_vs_post']:>9.3f}"
                      f"{bst['li_error_vs_pre']:>9.3f}{bst['legal_mass']:>8.4f}")